# ConvNeXt-Tiny: 세 전처리 × 3시드 비교

## 목적
기존 전처리에 마스킹·제외 사진 추가 활용을 붙였을 때의 결과와, 질감 전처리 결과를 비교합니다.
**시드 0의 15회가 끝나면 중간 비교표·공유 ZIP이 먼저 생성됩니다.** 이후 시드 1·2의 30회가 이어집니다.
모든 본 학습은 사용자가 Colab GPU(L4 권장)에서 실행합니다. 이 노트북의 GPU 학습은 아직 실행하지 않았습니다.

| 버전 | 학습 데이터와 입력 |
|---|---|
| v1 | 현환님 전처리 기존 통과 2,652장, 마스킹 없는 원본 고정 크롭 |
| v2 | 같은 기존 통과 사진의 승인된 마스크 + 형상 v2로 추가 활용한 743장, 총 3,395장 |
| v3_texture | 수화님 질감 v2 선별 + **수화님 원본 방식의 마스킹**, 아래 공통 면적 제외 정책 적용 |

**알고리즘을 섞지 않습니다.** v1/v2 마스크에 질감 v2를 새로 붙이지 않습니다.
질감 v2는 독립된 세 번째 비교 조건에만 사용합니다. 팀원 원본 코드·기존 데이터·기존 결과는 변경하지 않습니다.

## 평가와 면적 규칙
- 면적은 고정 크롭 1000×1000에서 남는 마스크 화소 수 / 1,000,000입니다. **10% 이하를 학습·검증·추론에서 제외**합니다.
- 기존 area_gt10_v3 제외 목록 61장은 세 조건 공통 제외합니다. 질감 버전은 자체 실제 마스크 면적 >10%와 원본 게이트 조건도 적용합니다.
- 질감 게이트는 확장 전 250×250 면적 >=10%, 실제 입력 면적은 테두리 확장 후 1000×1000 마스크 면적입니다. 둘은 다릅니다.
- 시간 블록 5-fold·30프레임 buffer, ImageNet V1, 입력512, 배치16, AdamW, 증강, 동일 업데이트 예산을 공유합니다.
- v1/v2는 같은3,395장으로 비교합니다. 세 조건 비교는 **셋 모두 입력 가능한 공통 사진**에서만 합니다.
- v1의 추가 활용 사진 예측, 질감 게이트가 보류하지만 마스크는 >10%인 사진의 예측은 **게이트를 우회한 진단**입니다.
- 각 조건의 통과율·보류율은 따로 표시합니다. 다른 통과 집합의 정확도로 직접 순위를 매기지 않습니다.
- 20라운드는 v1 학습 수를 기준으로 fold별 업데이트 수를 맞춘 것입니다. 데이터 수가 달라 전체 데이터 반복 횟수는 다릅니다.
- 마지막 가중치·확률0.5 고정. 검증 점수로 최적 에폭이나 기준값을 고르지 않습니다.

## 한계
클래스별 출력1회, 독립 테스트 없음, 회색 면적 지름길 미해결입니다.
형상 v2 저장 마스크는 오프라인 보정 정보를 사용했으며 새로운 출력의 실시간 동작은 검증되지 않았습니다.
세 시드 평균±표준편차는 초기화 변동이지 새로운 출력 일반화 신뢰구간이 아닙니다.
기존 수화님 1시드 결과는 평가 목록·학습 예산이 달라 이번 비교의 점수로 재사용하지 않습니다.


## 1. 설정
Drive `산공종설`에 기존 원본 ZIP, 기존 area_gt10_v3 데이터 ZIP, 이번 새 비교 ZIP을 둡니다.
기존 두 ZIP은 다시 업로드하지 않아도 됩니다. 새 `.ipynb`를 Colab에 업로드하고 GPU를 선택하세요.
설정·패키지 버전을 변경하면 RUN_NAME도 변경해야 합니다. 같은 설정이면 같은 RUN_NAME으로 재개할 수 있습니다.


In [ ]:
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/산공종설')
ORIGINAL_ZIP_NAME = 'underfill_originals_4000.zip'
BUNDLE_ZIP_NAME = 'resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip'
PATCH_ZIP_NAME = 'convnext_three_preproc_patch_20261005.zip'
RUN_NAME = 'convnext_three_preproc_20261005_run01'
IMG_SIZE = 512
BATCH = 16
ROUNDS = 20
SEEDS = [0, 1, 2]  # 시드0 → 중간 결과 → 시드1 → 중간 결과 → 시드2
FOLDS = [0, 1, 2, 3, 4]
RESCUE_MIN_AREA = 0.1
RUN_TRAINING = True  # False이면 입력 준비·점검만 실행
RUN_REMAINING_SEEDS = True  # 시드0 확인 후 멈추려면 False

RUN_MASK_DIAGNOSTICS = True
MASK_DIAG_PER_VERSION = 200


In [ ]:
import torch, torchvision
from google.colab import drive
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), '런타임 유형을 GPU로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name())
drive.mount('/content/drive')
OUTPUT = DRIVE_DIR / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)


## 2. 파일 준비·무결성 점검
원본과 마스크는 `/content` SSD에 풀고 학습 입력은 메모리맵으로 만듭니다. 512 입력 세 벌은 약8GiB입니다.
학습 결과·미완료 fold 체크포인트는 Drive에 보관합니다. 과거 결과를 삭제하지 않습니다.


In [ ]:
import zipfile, hashlib, json, shutil, subprocess, sys, os
import pandas as pd
from IPython.display import display
LOCAL = Path('/content/convnext_three_preproc_20261005')
LOCAL.mkdir(parents=True, exist_ok=True)
BUNDLE = LOCAL / 'bundle'
ORIGINALS = LOCAL / 'originals'
PATCH = LOCAL / 'patch'
TEXTURE = PATCH / 'texture'
CACHE = LOCAL / f'cache_{IMG_SIZE}'

def extract_zip(source, target):
    assert source.is_file(), f'파일 없음: {source}'
    local_zip = LOCAL / source.name
    if source.name != ORIGINAL_ZIP_NAME or not local_zip.exists() or local_zip.stat().st_size != source.stat().st_size:
        shutil.copy2(source, local_zip)
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as archive:
        for item in archive.infolist():
            assert (target/item.filename).resolve().is_relative_to(target.resolve()), 'ZIP 경로 오류'
        archive.extractall(target)
    print('준비:', source.name)

extract_zip(DRIVE_DIR / BUNDLE_ZIP_NAME, BUNDLE)
extract_zip(DRIVE_DIR / PATCH_ZIP_NAME, PATCH)
extract_zip(DRIVE_DIR / ORIGINAL_ZIP_NAME, ORIGINALS)
for folder in (BUNDLE, PATCH):
    hashes = json.loads((folder/'checksums.json').read_text())
    for name, expected in hashes.items():
        assert hashlib.sha256((folder/name).read_bytes()).hexdigest() == expected, name
    print('해시 점검:', folder.name, len(hashes), '개')
os.environ['PYTHONPATH'] = str(PATCH) + os.pathsep + str(BUNDLE) + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.path[:0] = [str(PATCH), str(BUNDLE)]
BASE_COMMAND = [sys.executable, '-u', str(PATCH/'train_convnext_three.py'),
    '--bundle', str(BUNDLE), '--texture', str(TEXTURE), '--originals', str(ORIGINALS),
    '--cache', str(CACHE), '--output', str(OUTPUT), '--size', str(IMG_SIZE),
    '--batch', str(BATCH), '--rounds', str(ROUNDS), '--rescue-min-area', str(RESCUE_MIN_AREA)]
subprocess.run(BASE_COMMAND + ['--verify-only'], check=True)
display(pd.read_csv(OUTPUT/'coverage.csv'))
display(pd.read_csv(OUTPUT/'eligibility_audit.csv'))
print('면적 정책:', json.loads((OUTPUT/'eligibility_policy.json').read_text()))


## 3. 실제 입력 미리보기
같은 사진의 원본 / 승인된 마스킹+추가 활용 / 질감 v2 마스킹을 확인합니다.
입력의 내부 RGB는 유지하고 바깥만RGB(124,116,104)로 칠합니다. 검토용 빨간 표시 이미지는 학습에 쓰지 않습니다.


In [ ]:
subprocess.run(BASE_COMMAND + ['--prepare-only'], check=True)
import numpy as np
import matplotlib.pyplot as plt
inputs = pd.read_csv(OUTPUT/'input_manifest.csv')
arrays = {name: np.load(CACHE/f'{name}.npy', mmap_mode='r') for name in ('raw', 'masked', 'texture')}
sample_indices = []
for cohort in ('primary', 'rescue'):
    for label in ('normal', 'underfill_50'):
        group = inputs[(inputs.cohort == cohort) & (inputs.label == label) & (inputs.texture_input == 1)]
        if len(group): sample_indices.append(group.index[len(group)//2])
fig, axes = plt.subplots(len(sample_indices), 3, figsize=(12, 4*len(sample_indices)), squeeze=False)
for row_no, idx in enumerate(sample_indices):
    r = inputs.iloc[idx]
    for col, name in enumerate(('raw', 'masked', 'texture')):
        axes[row_no, col].imshow(arrays[name][idx].transpose(1, 2, 0))
        axes[row_no, col].set_title(f"{name}: {r.label} #{r.frame_index} ({r.cohort})")
        axes[row_no, col].axis('off')
fig.tight_layout()
plt.show()
plt.close(fig)


## 4. 실행·중간 결과 표시 함수
로그는 Drive에 저장하고 화면에는 fold 완료 진행만 표시합니다.
라운드별로 재개 정보를 저장하므로 끊기면 현재 라운드 일부만 다시 진행합니다.
완료 fold의 모델 가중치는 보관하고, 더 이상 필요 없는 optimizer 재개 상태만 정리해 Drive 사용량을 줄입니다.
45개 모델 가중치는 대략5GB이며 입력 캐시는 Drive가 아니라 런타임 SSD에 있습니다.


In [ ]:
NAMES = {'v1': 'Original selection', 'v2': 'Masked + additional use', 'v3_texture': 'Texture v2 + mask'}

def export_results(suffix):
    destination = DRIVE_DIR / f'{RUN_NAME}_{suffix}_results.zip'
    temp = LOCAL / destination.name
    with zipfile.ZipFile(temp, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path.suffix.lower() in {'.csv', '.json', '.log', '.png', '.md'}:
                archive.write(path, path.relative_to(OUTPUT).as_posix())
    shutil.copy2(temp, destination)
    print('공유할 결과 ZIP:', destination)
    return destination

def show_results(seed):
    progress = json.loads((OUTPUT/'progress.json').read_text())
    assert seed in progress['fully_completed_seeds'], '이 시드의 15회가 아직 완료되지 않았습니다.'
    print(f'시드 {seed} 완료 — 전체 완료 {progress["completed_runs"]}/45회')
    for file, title in [('comparison_two.csv', '기존 통과 + 추가 활용: 두 조건의 같은 사진'),
                        ('comparison_three.csv', '세 전처리 모두 입력 가능한 같은 사진')]:
        print(title)
        table = pd.read_csv(OUTPUT/file)
        display(table[table.seed == seed][['version', 'cohort', 'n', 'accuracy', 'balanced_accuracy',
                                           'underfill_recall', 'normal_recall', 'auroc', 'fp', 'fn']])
    print('세 조건 공통 사진의 시드 평균·표준편차(1시드일 때 표준편차 없음)')
    means = pd.read_csv(OUTPUT/'across_seeds.csv')
    display(means[(means.scope == 'three') & means.metric.isin(['accuracy', 'underfill_recall', 'normal_recall'])])
    three = pd.read_csv(OUTPUT/'comparison_three.csv')
    three = three[three.seed == seed]
    groups = ['all_common_three', 'primary', 'rescue']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    colors = ['#315E8C', '#BF7A28', '#778449']
    for ax, group in zip(axes, groups):
        values = three[three.cohort == group].set_index('version').reindex(NAMES)
        for i, version in enumerate(NAMES):
            value = float(values.loc[version, 'accuracy'])
            ax.bar(i, value, color=colors[i], hatch=['', '//', '..'][i], edgecolor='#303030', linewidth=.6)
            ax.text(i, value+.015, f'{value:.1%}', ha='center', fontsize=10)
        ax.set_xticks(range(3), ['Original', 'Masked+add', 'Texture v2'])
        ax.set_title(f'{group}: same n={int(values.n.iloc[0])}')
        ax.set_ylim(0, 1.08)
        ax.set_yticks([0, .25, .5, .75, 1], ['0%', '25%', '50%', '75%', '100%'])
        ax.grid(axis='y', alpha=.2)
        ax.set_axisbelow(True)
    axes[0].set_ylabel('OOF accuracy')
    fig.suptitle(f'Seed {seed} — same evaluation photos; not an independent print test')
    fig.tight_layout()
    fig.savefig(OUTPUT/f'comparison_seed{seed}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return export_results(f'seed{seed}')

def train_seed(seed):
    log_path = OUTPUT / f'train_seed{seed}.log'
    command = BASE_COMMAND + ['--seeds', str(seed), '--folds', *map(str, FOLDS)]
    print(f'시드 {seed}: 3조건 × 5fold = 15회 시작. 로그: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as logfile:
        logfile.write('\n실행 명령: '+json.dumps(command, ensure_ascii=False)+'\n')
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace', bufsize=1)
        for line in process.stdout:
            logfile.write(line)
            logfile.flush()
            if f'round {ROUNDS}/{ROUNDS}:' in line or '이미 완료' in line:
                print(line.rstrip(), flush=True)
        return_code = process.wait()
    if return_code:
        tail = log_path.read_text(encoding='utf-8')[-7000:]
        print(tail)
        raise RuntimeError(f'학습 오류. 기존 저장은 유지됩니다. 로그: {log_path}')
    return show_results(seed)


## 5. 시드 0 먼저 실행 → 중간 결과
이 셀이 끝나면 비교표·그래프와 `..._seed0_results.zip`이 만들어집니다.
표가 나오기 전에는 학습 결과를 판단하지 않습니다. 시드0만 확인하고 잠시 멈추려면 이후 셀을 실행하지 마세요.
기존 기록은 fold당 약15분이었지만 GPU·고정 업데이트 수·캐시에 따라 달라집니다. 이번 실행 로그의 시간을 기준으로 보세요.


In [ ]:
assert SEEDS == [0, 1, 2] and FOLDS == [0, 1, 2, 3, 4], '기본 승인 범위는 3시드×5fold입니다.'
if RUN_TRAINING:
    SEED0_ZIP = train_seed(0)
else:
    print('입력 준비만 완료했습니다. 학습하지 않았습니다.')


## 5-1. 추가 학습 없이 마스크 민감도 점검
Drive에 진단용 addon ZIP을 추가한 뒤 실행합니다. 기존 가중치로 버전별 최대200장만 검사합니다.
색상 변경에서도 유효 화소·보간 경계는 고정합니다. 마스크 모양만의 입력은 보조 검사이며,
변화가 크거나 작다는 이유만으로 지름길의 존재/부재를 확정하지 않습니다.
이 셀은 기존 학습·전처리·분할·학습 횟수에 영향을 주지 않습니다.


In [ ]:
# 기존 노트북에도 이 셀을 시드0 완료 뒤 복붙할 수 있습니다. 학습 명령은 없습니다.
import hashlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display

MASK_DIAG_PER_VERSION = globals().get('MASK_DIAG_PER_VERSION', 200)
RUN_MASK_DIAGNOSTICS = globals().get('RUN_MASK_DIAGNOSTICS', True)
MASK_DIAG_ZIP_NAME = 'convnext_mask_diagnostics_addon_20261005.zip'

if RUN_MASK_DIAGNOSTICS:
    required = ['DRIVE_DIR', 'LOCAL', 'BUNDLE', 'TEXTURE', 'CACHE', 'OUTPUT', 'IMG_SIZE', 'BATCH']
    assert all(name in globals() for name in required), '기존 노트북의 설정·파일 준비 셀부터 실행하세요.'
    progress = json.loads((OUTPUT/'progress.json').read_text())
    assert 0 in progress['fully_completed_seeds'], '시드0의 15회 완료 후 실행하세요. 이 셀은 학습하지 않습니다.'
    ADDON = LOCAL/'mask_diagnostics_addon'
    ADDON.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_DIR/MASK_DIAG_ZIP_NAME) as archive:
        for member in archive.infolist():
            assert (ADDON/member.filename).resolve().is_relative_to(ADDON.resolve())
        archive.extractall(ADDON)
    for name, expected in json.loads((ADDON/'checksums.json').read_text()).items():
        assert hashlib.sha256((ADDON/name).read_bytes()).hexdigest() == expected, name
    DIAG_OUTPUT = OUTPUT/f'mask_diagnostics_seed0_n{MASK_DIAG_PER_VERSION}'
    command = [sys.executable, '-u', str(ADDON/'diagnose_mask_dependence.py'),
        '--bundle', str(BUNDLE), '--texture', str(TEXTURE), '--cache', str(CACHE),
        '--results', str(OUTPUT), '--output', str(DIAG_OUTPUT), '--size', str(IMG_SIZE),
        '--batch', str(BATCH), '--max-per-version', str(MASK_DIAG_PER_VERSION),
        '--seeds', '0', '--rescue-min-area', str(RESCUE_MIN_AREA), '--device', 'cuda']
    log_path = DIAG_OUTPUT/'diagnostics.log'
    DIAG_OUTPUT.mkdir(parents=True, exist_ok=True)
    print(f'추가 학습 없이 시드0의 v2 / v3_texture 각각 최대 {MASK_DIAG_PER_VERSION}장 추론 점검')
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n'+json.dumps(command, ensure_ascii=False)+'\n')
        process = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
    if process.returncode:
        print(log_path.read_text(encoding='utf-8')[-5000:])
        raise RuntimeError(f'점검 실패. 기존 학습 결과는 유지됩니다. 로그: {log_path}')
    summary = pd.read_csv(DIAG_OUTPUT/'summary.csv')
    print('색상 변경 가능 영역이 있는 사진의 판정 변화 (확률 변화는 0~1 단위)')
    display(summary[summary.scope == 'changeable_outside'][['version', 'arm', 'n',
        'flip_rate', 'mean_abs_probability_change', 'p95_abs_probability_change', 'accuracy', 'fp', 'fn']])
    print('전체 기존 OOF의 면적 관계: 인과관계나 지름길의 증명이 아님')
    area = pd.read_csv(DIAG_OUTPUT/'area_relations_oof.csv')
    display(area[area.scope.isin(['all_oof', 'normal', 'underfill_50'])])
    print('mask_only는 질감을 지운 분포 밖 입력의 보조 검사입니다. 색상 영향이 작아도 모양 의존이 없다는 뜻은 아닙니다.')
    print('층화 표본의 비율이며 전체 데이터의 성능 점수로 해석하지 마세요.')
    DIAG_ZIP = DRIVE_DIR/f'{RUN_NAME}_seed0_maskcheck_results.zip'
    temp_zip = LOCAL/DIAG_ZIP.name
    with zipfile.ZipFile(temp_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(DIAG_OUTPUT.rglob('*')):
            if path.is_file():
                archive.write(path, Path(DIAG_OUTPUT.name)/path.relative_to(DIAG_OUTPUT))
    shutil.copy2(temp_zip, DIAG_ZIP)
    print('점검 결과만 공유:', DIAG_ZIP)
else:
    print('마스크 점검을 건너뜁니다. 기존 학습·전처리는 변경하지 않습니다.')


## 6. 시드 1·2 실행 → 각각 결과 표시
전체 실행을 선택하면 시드0의 중간 결과를 먼저 출력한 뒤 이 셀을 이어서 실행합니다.
각 시드가 끝날 때 ZIP을 별도로 보관하므로 나중 실행이 끊겨도 앞선 결과는 남습니다.
실험 중 설정이나 데이터 묶음을 교체하지 마세요.


In [ ]:
if RUN_TRAINING and RUN_REMAINING_SEEDS:
    for seed in SEEDS[1:]:
        train_seed(seed)
    print('3시드 × 세 전처리 × 5fold = 45회 완료')
    FINAL_ZIP = export_results('all3seeds')
elif RUN_TRAINING:
    print('시드0 결과만 보관하고 나머지 시드 실행은 멈췄습니다.')


## 7. 결과 공유·해석
시드0 중간 결과는 `..._seed0_results.zip`, 최종은 `..._all3seeds_results.zip`을 공유해 주세요.
ZIP에는 OOF 확률, 설정, 입력 목록, 제외·통과율, 공통 사진 비교표, 시드 평균/표준편차, 로그가 들어 있습니다.
큰 모델 가중치는 결과 폴더에 남고 공유 ZIP에는 넣지 않습니다.
`summary.csv`의 all_available/pipeline_accepted는 버전마다 대상이 달라 직접 순위 비교하지 마세요.
v1/v2 전체 비교는 comparison_two.csv, 세 조건 비교는 comparison_three.csv입니다.
재활용 효과에는 학습 사진 추가와 마스킹 효과가 함께 섞여 있습니다. 이 두 요인의 분리 효과는 이번 세 조건만으로 확정할 수 없습니다.


In [ ]:
if (OUTPUT/'progress.json').exists():
    print(json.loads((OUTPUT/'progress.json').read_text()))
    print('결과 폴더:', OUTPUT)
    print('시드0 ZIP:', DRIVE_DIR/f'{RUN_NAME}_seed0_results.zip')
    print('전체 ZIP:', DRIVE_DIR/f'{RUN_NAME}_all3seeds_results.zip')


## 8. 실행 종료 후 런타임 반납
Drive 결과 ZIP이 있는지 확인한 뒤 실행합니다. 다음 셀은 런타임 연결을 끝냅니다.
학습 중에는 실행하지 마세요. 나중에 재개할 때는 같은 환경·설정·RUN_NAME으로 위부터 실행합니다.


In [ ]:
from google.colab import runtime
drive.flush_and_unmount()
runtime.unassign()
